# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an end-to-end workflow for exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. 

### Dataset Source
The dataset metadata is provided via a Croissant schema URL.


In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This includes inspecting the dataset summary and ensuring connectivity to the source.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the URL to the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Dataset entities (record sets, fields, and columns) are referenced by their unique `@id` values.

Let's enumerate available record sets and their fields. 
This will help us identify which data slices to load for further analysis.

In [ ]:
# List available record sets and their fields (by `@id`).
record_sets = list(dataset.record_sets.keys())
print("Available record set @ids:")
for rs_id in record_sets:
    print(f"  - {rs_id}")
    record_set = dataset.record_sets[rs_id]
    fields = getattr(record_set, 'fields', [])
    if fields:
        print("    Fields (@id):")
        for field in fields:
            print(f"      - {field['@id']}")
    else:
        print("    No fields found.")

## 3. Data Extraction

Now we select the dataset's tabular record set(s) to extract records for detailed analysis.

- **Use each record set's `@id`.**
- Store each record set's data in a named DataFrame.
- Display the first columns and a preview of the data for one record set.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set: {record_set_id}\nColumns: {df.columns.tolist()}\nSample records:")
    display(df.head(3))

# For the main tabular data, select the first record set for further analysis
main_record_set_id = record_sets[0] if record_sets else None
main_df = dataframes[main_record_set_id] if main_record_set_id else None
if main_df is not None:
    print("\nMain DataFrame preview:")
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)

Let's practice some standard data analysis steps, referencing all entities by their Croissant `@id` values.

- Filter records based on a numeric field (e.g., age)
- Normalize the numeric field
- Group and aggregate by a key attribute

Adjust the code below to select the appropriate field `@id` for the numeric and group fields. If you are not sure, use the field listings from Section 2.

In [ ]:
# Choose a numeric field (e.g., '@id' for 'Age'); adjust these to your dataset
numeric_field_id = None
group_field_id = None

# Attempt to choose appropriate field IDs from the main DataFrame columns
if main_df is not None:
    for col in main_df.columns:
        if 'age' in col.lower():
            numeric_field_id = col
        if ('sex' in col.lower()) or ('gender' in col.lower()):
            group_field_id = col

if numeric_field_id is None:
    # Provide as example the first numeric dtype field
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break
if group_field_id is None and len(main_df.columns) > 1:
    group_field_id = main_df.columns[1]

print(f"Selected numeric field: {numeric_field_id}")
print(f"Selected group field: {group_field_id}")

# Remove outliers: keep only records with numeric_field > threshold
threshold = 10
if numeric_field_id in main_df.columns:
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
    display(filtered_df.head())
    
    # Normalize selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (first 5 rows):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by group_field if present
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped statistics by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df)

## 5. Visualization

Visualize distributions and relationships:
- Histogram of the numeric field (e.g., Age)
- Boxplot of the numeric field grouped by a categorical field (e.g., Sex or Cancer Type)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot grouped by group_field, if available
    if group_field_id in main_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

- We have successfully loaded the FAIR^2 dataset from its Croissant schema using its canonical `@id` references for all record sets and fields.
- Through basic EDA and visualization, we can begin to characterize the distribution and relationships of key clinicopathological variables within this important cancer survivor cohort.

_To extend this analysis, consider exploring additional fields, advanced visualizations, or machine learning workflows tailored to biomedical questions of MSI-H prevalence and anatomical distribution._